In [ ]:
import findspark
findspark.init()

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import current_date, date_sub, year

spark = SparkSession.builder. \
    appName("pyspark-1"). \
    getOrCreate()

In [ ]:
'''1. Deployment : I would like to deploy my code cloud or an on prem cluster. I would push my code to git for version control 
purpose and likewise i can place my py file in s3 or on prem cluster. Apart from that i can also run my 
code using notebook but that will not be a very scable approach as my code base can grow in future'''
'''2. Trigger : To trigger my code i would use client method as it would be easier to debug and since my job
is not running for longer duration of time so client mode is preferred in this case'''

### Read data

In [61]:
import os
from pyspark.sql.functions import *

# print(os.getcwd())
df = spark.read.csv("/data_engineering_takehome1-main/dataset/nyc-jobs.csv", header=True)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

### Sample function

In [14]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [75]:
##below function is for feature engineering transformations
#1. add a column for average salary
#2. add a process date feature
#3. add process year

def feature_engineering(df: DataFrame) -> list:
    df = df.withColumn("avg_salary",round((col("Salary Range From") + col("Salary Range To")) / 2, 2))\
         .withColumn("Process Date feature", to_date("Process Date")) \
         .withColumn("Process Date feature", to_date(to_timestamp("Process Date feature", "yyyy-MM-dd'T'HH:mm:ss.SSS")))\
          .withColumn("process_year", year("Process Date feature"))
    
    return df

df = feature_engineering(df)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

In [87]:
##this function is for writing the data to a location in csv format, it takes necessary arguments like
#1. dataframe
#2. mode of writing file 
#3. number of repartition files
#4. path to write file

def write_csv(df: DataFrame, mode, n, path):
    df.repartition(n).write.csv(path, header=True, mode=mode)

In [72]:
#test block to get overview of dataframe

df.show(1,False,True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result

In [97]:
#this function gets us number of jobs posting per category

def get_number_of_job_postings_per_category(df: DataFrame, n: int) -> list:
    row_df = df.filter(df["Job Category"].isNotNull())\
    .groupBy('Job Category').agg(sum("# Of Positions").alias("total_job_postings")) \
           .orderBy("total_job_postings", ascending=False).limit(n)
    return row_df

job_postings_per_category_df = get_number_of_job_postings_per_category(df, 10)
write_csv(job_postings_per_category_df, 'overwrite', 1, '/data_engineering_takehome1-main/jupyter/output/number_of_job_postings_per_category')
job_postings_per_category_df.show(100,False)



+-----------------------------------------+------------------+
|Job Category                             |total_job_postings|
+-----------------------------------------+------------------+
|Public Safety, Inspections, & Enforcement|1407.0            |
|Building Operations & Maintenance        |1249.0            |
|Engineering, Architecture, & Planning    |762.0             |
|Legal Affairs                            |515.0             |
|Technology, Data & Innovation            |405.0             |
|Health                                   |358.0             |
|Administration & Human Resources         |330.0             |
|Finance, Accounting, & Procurement       |275.0             |
|Maintenance & Operations                 |212.0             |
|Policy, Research & Analysis              |200.0             |
+-----------------------------------------+------------------+



In [89]:
#this function gets us salary distribution per category


def get_salary_distribution_per_category(df: DataFrame) -> list:
    row_df = df.filter(df["Job Category"].isNotNull())\
    .groupBy('Job Category').agg(round(min("avg_salary"), 2).alias("min_salary"),
          round(max("avg_salary"), 2).alias("max_salary"),
          round(avg("avg_salary"), 2).alias("avg_salary")) \
           .orderBy("avg_salary", ascending=False)
    return row_df

salary_distribution_per_category_df = get_salary_distribution_per_category(df)
write_csv(salary_distribution_per_category_df, 'overwrite', 1, '/data_engineering_takehome1-main/jupyter/output/salary_distribution_per_category')

salary_distribution_per_category_df.show(100,False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+----------+
|Job Category                                                                                                                                                                                             |min_salary|max_salary|avg_salary|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------+----------+
|Administration & Human Resources Finance, Accounting, & Procurement Building Operations & Maintenance                                                                                                    |218587.0  |218587.0  |218587.0  |
|Engineering, Architecture, & Planning Maintenance &

In [90]:
#this function gets us salary distribution per agency


def get_salary_distribution_per_agency(df: DataFrame) -> list:
    
    window = Window.partitionBy("Agency").orderBy(col("Salary Range To").desc())
    df = df.select('Agency','Salary Range To','Business Title')
    df_rank = df.withColumn("rank", row_number().over(window)).filter(col('rank') == 1).drop('rank')
    
    
    
    return df_rank

salary_distribution_per_agency_df = get_salary_distribution_per_agency(df)
write_csv(salary_distribution_per_agency_df, 'overwrite', 1, '/data_engineering_takehome1-main/jupyter/output/salary_distribution_per_agency')

salary_distribution_per_agency_df.show(100,False)

+------------------------------+---------------+--------------------------------------------------------------------+
|Agency                        |Salary Range To|Business Title                                                      |
+------------------------------+---------------+--------------------------------------------------------------------+
|ADMIN FOR CHILDREN'S SVCS     |98163          |Network Engineer I LAN/WAN                                          |
|ADMIN TRIALS AND HEARINGS     |42799          |Facilities & Security Coordinator                                   |
|BOARD OF CORRECTION           |65625          |Standards Specialist (Monitor)                                      |
|BOROUGH PRESIDENT-QUEENS      |95270          |Engineering                                                         |
|BUSINESS INTEGRITY COMMISSION |85000          |Computer Systems Manager                                            |
|CIVILIAN COMPLAINT REVIEW BD  |61936          |Administ

In [91]:
#this function gets us salary per agency for last 2 years


def get_salary_per_agency_last2_years(df: DataFrame) -> list:
    
    df_agency_last2yrs = df.filter(col("Process Date feature") >= date_sub(current_date(), 365*2))
    df_agency_last2yrs.show()
    avg_salary_per_agency_last2 = (
    df_agency_last2yrs.groupBy("Agency")
                .agg(round(avg("avg_salary"), 2).alias("avg_salary"))
                .orderBy("avg_salary", ascending=False)
)

    
    
    return avg_salary_per_agency_last2

salary_per_agency_last2_years_df = get_salary_per_agency_last2_years(df)
write_csv(salary_per_agency_last2_years_df, 'overwrite', 1, '/data_engineering_takehome1-main/jupyter/output/salary_per_agency_last2_years')

salary_per_agency_last2_years_df.show(100,False)

+------+------+------------+--------------+--------------+-------------------+-------------+-----+------------+-----------------------------+-----------------+---------------+----------------+-------------+------------------+---------------+-------------------------+----------------+----------------------+--------+-----------+---------------+-------------------+---------------------+------------+----------+---------------+------------+----------+--------------------+------------+
|Job ID|Agency|Posting Type|# Of Positions|Business Title|Civil Service Title|Title Code No|Level|Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|Work Location|Division/Work Unit|Job Description|Minimum Qual Requirements|Preferred Skills|Additional Information|To Apply|Hours/Shift|Work Location 1|Recruitment Contact|Residency Requirement|Posting Date|Post Until|Posting Updated|Process Date|avg_salary|Process Date feature|process_year|
+------+------+------------+--

In [95]:
##this function gets us highest paid skills in market


def get_highest_paid_skill(df: DataFrame, top_n: int) -> list:
    
    window = Window.orderBy(col("Salary Range To").desc())
    df = df.select('Salary Range To','Business Title').distinct()
    df_rank = df.withColumn("rank", row_number().over(window)).distinct()
    
    df_top = df_rank.filter(col("rank") <= top_n).drop("rank").distinct()
    
    
    
    return df_top

highest_paid_skill_df = get_highest_paid_skill(df, 10)
write_csv(highest_paid_skill_df, 'overwrite', 1, '/data_engineering_takehome1-main/jupyter/output/highest_paid_skill')

highest_paid_skill_df.show(100,False)

26/02/17 18:57:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 1

+---------------+--------------------------------------------------------------------+
|Salary Range To|Business Title                                                      |
+---------------+--------------------------------------------------------------------+
|99406          |Senior Windows Administrator                                        |
|99394          |PROCUREMENT/NEGOTIATIONS ATTORNEY                                   |
|99394          |Agency Attorney                                                     |
|99303          |Assistant Project Manager                                           |
|99161          |COMPUTER SPECIALIST (SOFTWARE)                                      |
|99000          |Senior Policy Advisor, Climate Partnerships                         |
|98908          |Environmental Health & Safety Incident Investigator                 |
|98908          |Integrated Pest Management Oversight Team Specialist                |
|98818.92       |Business/Data Analyst, Bur

26/02/17 18:57:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/17 18:57:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
